# L6 · Extremo a extremo
El ejemplo precargado → comparación real (LLM y embeddings del stage, FAISS real) → papel de trabajo Excel, en una sola corrida. Es el mismo servicio que usa la app (`backend/engine/servicio.py`).

**Cambia aquí:** `N_ARTICULOS` (cuántos artículos del ejemplo comparar) y `N_SECCIONES` (cuántas secciones del manual). Con Groq gratuito cada par tarda ~10 s; con Foundry, mucho menos.

In [ ]:
import os, sys, time
from pathlib import Path
os.environ.setdefault("STAGE", "sandbox")
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from backend.core.progreso import configurar_logs
configurar_logs()
from backend.config import settings
from backend.core import sesiones
from backend.engine import servicio
from backend.llm.client import ModelClient
print("stage:", os.environ["STAGE"], "| LLM:", settings.LLM_PROVIDER, "| embeddings:", settings.EMB_PROVIDER)

N_ARTICULOS = 4     # artículos del ejemplo (el ejemplo trae 18: Art. 31 a 48)
N_SECCIONES = 8     # secciones del manual (el ejemplo trae 16 comparables)

In [ ]:
cliente = ModelClient()
print(cliente.ping())          # LLM y embeddings reales responden
s = sesiones.crear()
docs = servicio.cargar_ejemplo(s)
print([(d.id, d.nombre[:30], d.tipo, d.estado) for d in docs])

def hojas(nodos):
    for n in nodos:
        if n.hoja: yield n
        yield from hojas(n.hijos)
arts = [h.id for h in hojas(servicio.arbol(s, "normativa")[0].nodos)][:N_ARTICULOS]
secs = [h.id for h in hojas(servicio.arbol(s, "manual_control")[0].nodos) if h.seleccionada][:N_SECCIONES]
print(len(arts), "artículos x", len(secs), "secciones")

In [ ]:
t0 = time.time()
job = servicio.comparar(s, arts, secs, cliente)
from tqdm.auto import tqdm
barra = tqdm(total=100, desc="Comparación", unit="%")
while job.estado in ("en_cola", "en_curso"):
    barra.n = int(job.progreso * 100); barra.set_postfix_str(f"{job.etapa} {job.mensaje}"); barra.refresh(); time.sleep(2)
barra.n = 100; barra.close()
print(job.estado, f"{time.time() - t0:.0f} s", job.errores)
assert job.estado == "listo"

In [ ]:
comp = s.comparaciones[job.comparacion_id]
r = comp.out
print("conteos:", r.conteos, "| pares juzgados:", r.pares_juzgados, "| a revisión:", r.pares_a_revision, "| avisos:", r.avisos)
import pandas as pd
pd.DataFrame([{"Artículo": f.identificador, "Marca": f.marca, "Respaldo": f.seccion_respaldo, "Confianza": f.confianza, "Revisar": f.requiere_revision, "Comentario": f.comentario[:90]} for f in r.via1])

In [ ]:
print("controles sin base normativa:", [f.identificador for f in r.via2 if f.sin_base])
print("\n" + r.conclusion)

In [ ]:
ruta = servicio.excel(s, comp)
import shutil
SALIDA = RAIZ / "output"; SALIDA.mkdir(exist_ok=True)       # carpeta ignorada por git
destino = shutil.copy(ruta, SALIDA / "papel_e2e.xlsx")
from openpyxl import load_workbook
wb = load_workbook(destino)
print("Excel:", destino, wb.sheetnames, "| filas Hoja 1:", wb["Papel de trabajo"].max_row, "| filas Hoja 2:", wb["Anexo técnico"].max_row)
sesiones.reiniciar(s.id)